In [ ]:
import json, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
PROJECT_DIR = '/content/drive/MyDrive/deepcare'
TARGET = 'Diabetes_binary'

df = pd.read_csv(f'{PROJECT_DIR}/data/raw.csv').drop_duplicates().reset_index(drop=True)
print(df.shape, round(df[TARGET].mean(), 4))   # expect (229474, 22) and ~0.1529

In [ ]:
X, y = df.drop(columns=[TARGET]), df[TARGET]
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

for name, part in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(name, len(part), round(part.mean(), 4))

In [ ]:
NUM = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
BIN = [c for c in X.columns if c not in NUM]

lo, hi = X_train['BMI'].quantile([0.005, 0.995])

def clip_bmi(d):
    d = d.copy(); d['BMI'] = d['BMI'].clip(lo, hi); return d

scaler = StandardScaler().fit(clip_bmi(X_train)[NUM])

def transform(d):
    d = clip_bmi(d); out = d.copy()
    out[NUM] = scaler.transform(d[NUM])
    return out[BIN + NUM]          # fixed column order, reused by the demo app

Xtr, Xva, Xte = transform(X_train), transform(X_val), transform(X_test)
print(Xtr[NUM].agg(['mean', 'std']).round(3))   # train: mean ~0, std ~1

In [ ]:
rows = lambda Xd, yd: set(map(tuple, pd.concat([Xd, yd], axis=1).values))
a, b, c = rows(X_train, y_train), rows(X_val, y_val), rows(X_test, y_test)
print('overlap train&val:', len(a & b), '| train&test:', len(a & c), '| val&test:', len(b & c))  # expect 0

d = f'{PROJECT_DIR}/data'
for name, Xs, Xr, ys in [('train', Xtr, X_train, y_train), ('val', Xva, X_val, y_val), ('test', Xte, X_test, y_test)]:
    pd.concat([Xs, ys], axis=1).to_csv(f'{d}/{name}_scaled.csv', index=False)
    pd.concat([Xr, ys], axis=1).to_csv(f'{d}/{name}_raw.csv', index=False)

joblib.dump(scaler, f'{PROJECT_DIR}/scaler.joblib')
json.dump({'NUM': NUM, 'BIN': BIN, 'bmi_clip': [float(lo), float(hi)], 'seed': SEED},
          open(f'{PROJECT_DIR}/prep_config.json', 'w'))